# MoE LiDAR Detection — Router Training & Evaluation

This notebook trains and evaluates the router models at the core of this
project's output-level Mixture-of-Experts pipeline: five frozen, pretrained
LiDAR 3D detectors (CenterPoint-Voxel, CenterPoint-Pillar, PointPillars, SSN,
BEVFusion-LiDAR) are fused by a learned per-class scorer. No expert is
fine-tuned here — this notebook trains only the router.

**What this notebook does, in order:**

1. Load the pre-built router training dataset (`training_data/router_data/`).
2. Train a per-class XGBoost router (bicycle additionally sigmoid-calibrated).
3. Train a per-class FT-Transformer router.
4. Evaluate both routers — and their per-class ensemble blend — on the
   held-out calibration split, matching the metrics reported in the paper.
5. *(Optional, advanced)* Run the complete pipeline — ensemble blend, NMS,
   temporal refinement — on raw expert predictions and score it with the
   official nuScenes evaluator, reproducing the project's headline
   mAP/NDS numbers.

**Before running:** download `train.csv` and `eval.csv` from the Google
Drive link in [`training_data/README.md`](../training_data/README.md) and
place them at `training_data/router_data/`. Steps 1–4 need nothing else.
Step 5 additionally needs the expert prediction JSONs
([`predictions/README.md`](../predictions/README.md)) and a nuScenes
metadata download — see that section for details before running it.


## 0. Setup

**Memory note (GB10 / unified memory).** GPU and system RAM come out of one
shared 121 GB pool here, so a large allocation can take the whole machine down
rather than raising a clean `MemoryError`.

Measured peaks on this machine: router training (Steps 2–4) is cheap — about
1.2 GB on the GPU at the configured batch size of 4096. The expensive part is
Section 7, on the host side, where nuScenes metadata (~7.7 GB) and the dilated
drivable-area map masks (~23.9 GB peak) dominate. Earlier reboots were
originally blamed on the FT-Transformer batch size; measurement ruled that out
(see the comment in the next cell), and Section 7's host-side spike landing on
top of retained training state is the likelier culprit.

Because of that, the knobs below gate the *optional* sections instead of slowing
down training. Keep `RUN_FULL_PIPELINE` and `RUN_CV_DIAGNOSTIC` off until
Steps 2–4 succeed, then enable one at a time — ideally after a kernel restart, so
Section 7's spike doesn't stack on top of everything Steps 2–4 still hold.
Each `free_memory(...)` call prints the running peak, so if the machine does die
you can see which step it died after.

In [ ]:
import sys
import gc
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(REPO))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.moe.features import FEATURE_NAMES
from src.moe.xgboost_router import (
    NUSCENES_CLASSES,
    train_xgboost_per_class_routers,
    save_xgboost_per_class_routers,
    load_xgboost_per_class_routers,
)
from src.moe.nn_router import (
    save_nn_per_class_routers,
    train_nn_per_class_routers,
)
# ── Memory / device knobs (GB10: GPU and system RAM
# share one 121 GB pool) ───────────────────────────

# Host-side costs, measured on this machine:
#
# 5 expert JSONs ... 1.9 GB filtered to 804 tokens
#                  (3.9 GB unfiltered)
# nuScenes meta  ... 7.7 GB
# warm_map_masks .. 23.9 GB peak
#                  <-- largest host allocation
#                  Section 7 only
#
# GPU side, now measured instead of guessed.
# One per-class FT-Transformer, forward + backward
# + Adam, peak torch allocation on this 130.7 GB
# device:
# batch 1024 ... 0.32 GB
# batch 4096 ... 1.18 GB
#
# A 0.9 GB difference cannot reboot this machine,
# so batch size was not the cause of earlier crashes.
# The host-side numbers above are the likely cause:
# Section 7's ~24 GB dilation landed on top of
# everything training still held.
#
# Keep 4096: configs/moe_final.yaml records it, and
# batch size changes which rows land in each step.
# Training at 1024 yields genuinely different routers:
# measured mAP 0.6157 at 1024 vs 0.6179 at 4096.
# the documented path; see xgboost_router.py docstring
USE_GPU_XGB = True           
USE_GPU_NN = True
# = ft_transformer.batch_size in configs/moe_final.yaml
NN_BATCH_SIZE = 4096         
# On unified memory, a runaway GPU allocation can
# take the OS with it instead of raising OOM.
# Capping the process share converts that into a
# normal Python exception that can be caught, which
# is worth the small loss of headroom.
# None to disable
GPU_MEMORY_FRACTION = 0.5    
# 5x XGBoost retrain; enable only when debugging overfit
RUN_CV_DIAGNOSTIC = False    
# nuScenes + map masks (~30 GB); run last, alone
RUN_FULL_PIPELINE = False    

# Which frames Section 7 scores. "calibration"
# (804 frames, = eval.csv) is a fast sanity check,
# but it is the split used to tune thresholds and
# lambdas, and it is unusually short on
# construction_vehicle and barrier, so its mAP
# reads low (~0.53).
#
# The documented mAP=0.6179 / NDS=0.6761 is on
# "test" (1,804 held-out frames). Use that split
# to compare against the reported numbers.

# "calibration" | "test"
EVAL_SPLIT = "test"   

def free_memory(label: str = "") -> None:
    """
        Drop garbage, release cached GPU blocks, and report
        peak memory so far.

        ru_maxrss is a high-water mark for the whole kernel,
        so it keeps rising. That is exactly what's wanted to
        find which step precedes a crash.
    """
    import resource

    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

    peak_gb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6
    print(f"[mem] peak RSS {peak_gb:.2f} GB{f' — {label}' if label else ''}")

    # Printed output dies with the machine, which is
    # precisely when knowing how far the run got matters
    # most, so mirror each checkpoint to disk (fsynced).
    #
    # Pair with `python scripts/hw_monitor.py &` for
    # continuous samples between these markers, and read
    # both back with `--summary`.
    try:
        from scripts.hw_monitor import mark
        mark(label or "checkpoint")
    except Exception:
        pass

def cap_gpu_memory(fraction: float | None = GPU_MEMORY_FRACTION) -> None:
    """Bound this process's GPU allocation so exhaustion 
    raises instead of rebooting."""
    if fraction is None:
        return
    try:
        import torch
        if not torch.cuda.is_available():
            return
        torch.cuda.set_per_process_memory_fraction(fraction, 0)
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(
            f"[gpu] capped at {fraction:.0%} of {total:.1f} GB "
            f"= {fraction * total:.1f} GB"
        )
    except Exception as exc:  # older/odd builds may not support the call
        print(f"[gpu] could not set memory cap: {type(exc).__name__}: {exc}")

cap_gpu_memory()

pd.set_option("display.width", 120)
print(f"Repo root: {REPO}")
print(f"Feature vector ({len(FEATURE_NAMES)} features): {FEATURE_NAMES}")
print(
    f"Memory knobs: USE_GPU_XGB={USE_GPU_XGB} USE_GPU_NN={USE_GPU_NN} "
    f"NN_BATCH_SIZE={NN_BATCH_SIZE} RUN_CV={RUN_CV_DIAGNOSTIC} "
    f"RUN_FULL_PIPELINE={RUN_FULL_PIPELINE} EVAL_SPLIT={EVAL_SPLIT!r}"
)
print(
    "Reminder: run Steps 2-4, then Section 7, in SEPARATE kernel sessions. "
    "Step 3 saves weights to model_weights/, so Section 7 can reload them "
    "instead of keeping training state resident."
)

# Opening marker, so outputs/hw_monitor.csv has a clean boundary 
# showing where
# this run began rather than blending into the previous one.
free_memory("Step 0 — setup complete, run starting")


[gpu] capped at 50% of 130.7 GB = 65.3 GB
Repo root: /home/santoshaibox/msaai/capstone/code/direct-data-code/moe-lidar-detection
Feature vector (17 features): ['expert_id', 'detection_score', 'dist_from_ego', 'box_width', 'box_length', 'box_height', 'vel_magnitude', 'n_peer_overlaps', 'max_peer_iou', 'mean_peer_score', 'score_variance', 'expert_agreement', 'n_spatial_overlaps', 'class_agreement', 'max_class_score', 'n_active_experts', 'dist_to_drivable_area']
Memory knobs: USE_GPU_XGB=True USE_GPU_NN=True NN_BATCH_SIZE=4096 RUN_CV=False RUN_FULL_PIPELINE=False EVAL_SPLIT='test'
Reminder: run Steps 2-4, then Section 7, in SEPARATE kernel sessions. Step 3 saves weights to model_weights/, so Section 7 can reload them instead of keeping training state resident.


## 1. Load the router training dataset

`train.csv` (3,411 keyframes / 85 scenes) is used to fit both routers.
`eval.csv` (804 keyframes / 20 scenes) is a disjoint, scene-grouped
calibration split used only for evaluation in this notebook — never for
fitting model parameters. See `configs/moe_final.yaml` → `split` for how
these partitions were built.


In [ ]:
DATA_DIR = REPO / "training_data" / "router_data"

# Read the feature columns straight into float32 (~halves RAM for 2M rows).
# Done via read_csv's dtype rather than a post-hoc .astype(), which would
# briefly hold both the float64 and float32 copies at once.
_header = pd.read_csv(DATA_DIR / "train.csv", nrows=0).columns
_dtypes = {c: np.float32 for c in FEATURE_NAMES if c in _header}

train_df = pd.read_csv(DATA_DIR / "train.csv", dtype=_dtypes)
eval_df = pd.read_csv(DATA_DIR / "eval.csv", dtype=_dtypes)


print(
    f"train.csv: {len(train_df):,} rows across "
    f"{train_df['sample_token'].nunique():,} keyframes"
)

print(
    f"eval.csv:  {len(eval_df):,} rows across "
    f"{eval_df['sample_token'].nunique():,} keyframes"
)
train_df.head()


train.csv: 1,723,052 rows across 3,411 keyframes
eval.csv:  371,102 rows across 804 keyframes


,expert_id,class_id,detection_score,dist_from_ego,box_width,box_length,box_height,vel_magnitude,n_peer_overlaps,max_peer_iou,...,score_variance,expert_agreement,n_spatial_overlaps,class_agreement,max_class_score,n_active_experts,dist_to_drivable_area,label,sample_token,model_name
0,6.0,0.0,0.789173,975.066833,1.874180,4.396540,1.627261,0.0,8.0,0.951364,...,0.112497,1.0,8.0,0.625000,0.92454,5.0,0.0,1,0046092508b14f40a86760d11f9896bb,bevfusion_lidar
1,6.0,0.0,0.687098,953.779419,1.823229,4.433558,1.545039,0.0,7.0,0.957836,...,0.092258,1.0,7.0,0.571429,0.92454,5.0,0.0,1,0046092508b14f40a86760d11f9896bb,bevfusion_lidar
2,6.0,0.0,0.705819,967.571594,1.873642,4.500848,1.521493,0.0,5.0,0.974127,...,0.102421,1.0,5.0,0.800000,0.92454,5.0,0.0,1,0046092508b14f40a86760d11f9896bb,bevfusion_lidar
3,6.0,8.0,0.421172,959.523743,0.350162,0.350411,0.794157,0.0,8.0,0.816722,...,0.021778,1.0,8.0,0.625000,0.67420,5.0,0.0,1,0046092508b14f40a86760d11f9896bb,bevfusion_lidar
4,6.0,8.0,0.434399,965.557434,0.336308,0.330591,0.725230,0.0,4.0,0.707611,...,0.033106,1.0,4.0,1.000000,0.67420,5.0,0.0,1,0046092508b14f40a86760d11f9896bb,bevfusion_lidar


In [ ]:
# ── Run-mode overrides ───────────────────────────────────────────────────────
# Single place to control which optional sections run, so the Setup cell's
# defaults stay as documented. After a kernel restart, re-run the Setup cell
# and then this one — together they are all Section 7 needs when
# EVAL_SPLIT="test" (routers and config are reloaded from disk).
# Section 7: full pipeline + official nuScenes eval
RUN_FULL_PIPELINE = True    
# 1,804 held-out frames — comparable to mAP 0.6179
EVAL_SPLIT = "test"         
# Section 6 only; costs 5 extra XGBoost retrains
RUN_CV_DIAGNOSTIC = False   

print(f"Run mode: RUN_FULL_PIPELINE={RUN_FULL_PIPELINE} "
      f"EVAL_SPLIT={EVAL_SPLIT!r} RUN_CV_DIAGNOSTIC={RUN_CV_DIAGNOSTIC}")

Run mode: RUN_FULL_PIPELINE=True EVAL_SPLIT='test' RUN_CV_DIAGNOSTIC=False


## 2. Train per-class XGBoost routers

One `XGBClassifier` per nuScenes class (200 rounds, depth 6, learning rate
0.05, `scale_pos_weight` for imbalance). Bicycle — the class with the most
extreme imbalance (~2% positive) — is additionally wrapped in
`CalibratedClassifierCV` (sigmoid/Platt scaling) to correct probability-scale
distortion that `scale_pos_weight` introduces; see the module docstring in
`src/moe/xgboost_router.py` for why this matters only for that one class.

Training runs on GPU if available; the fitted models are always reset to
CPU before saving, since this pipeline's inference-time batches are small
enough that GPU prediction is *slower* due to a mismatched-device fallback.


In [ ]:
xgb_routers = train_xgboost_per_class_routers(
    train_df, val_df=eval_df, 
    feature_names=FEATURE_NAMES, use_gpu=USE_GPU_XGB,
)

XGB_OUT = REPO / "model_weights" / "router_xgboost"
save_xgboost_per_class_routers(xgb_routers, XGB_OUT)
free_memory("after XGBoost training")


2026-08-02 09:59:19 | INFO     | src.moe.xgboost_router | Training XGBoost router per class on cuda | features=17
2026-08-02 09:59:21 | INFO     | src.moe.xgboost_router | Trained car                   : 363530 rows, 33.03% positive, n_estimators=152
2026-08-02 09:59:21 | INFO     | src.moe.xgboost_router |   └─ val AUC=0.9775  AP=0.9637
2026-08-02 09:59:21 | INFO     | src.moe.xgboost_router | Trained truck                 : 141284 rows, 12.67% positive, n_estimators=120
2026-08-02 09:59:21 | INFO     | src.moe.xgboost_router |   └─ val AUC=0.9599  AP=0.7825
2026-08-02 09:59:22 | INFO     | src.moe.xgboost_router | Trained construction_vehicle  : 75450 rows, 3.83% positive, n_estimators=29
2026-08-02 09:59:22 | INFO     | src.moe.xgboost_router |   └─ val AUC=0.9042  AP=0.0788
2026-08-02 09:59:22 | INFO     | src.moe.xgboost_router | Trained bus                   : 26923 rows, 15.96% positive, n_estimators=11
2026-08-02 09:59:22 | INFO     | src.moe.xgboost_router |   └─ val AUC=0.991

/home/santoshaibox/envs/sdk_16_2/lib/python3.12/site-packages/xgboost/core.py:553: UserWarning: [09:59:24] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


2026-08-02 09:59:25 | INFO     | src.moe.xgboost_router | Trained bicycle               : 179394 rows, 2.18% positive, n_estimators=159 (calibrated)
2026-08-02 09:59:25 | INFO     | src.moe.xgboost_router |   └─ val AUC=0.9312  AP=0.6848
2026-08-02 09:59:26 | INFO     | src.moe.xgboost_router | Trained pedestrian            : 413558 rows, 17.54% positive, n_estimators=75
2026-08-02 09:59:26 | INFO     | src.moe.xgboost_router |   └─ val AUC=0.9585  AP=0.7698
2026-08-02 09:59:26 | INFO     | src.moe.xgboost_router | Trained traffic_cone          : 193577 rows, 6.72% positive, n_estimators=32
2026-08-02 09:59:26 | INFO     | src.moe.xgboost_router |   └─ val AUC=0.9562  AP=0.5234
2026-08-02 09:59:26 | INFO     | src.moe.xgboost_router | Saved XGBoost per-class routers for 10 classes -> /home/santoshaibox/msaai/capstone/code/direct-data-code/moe-lidar-detection/model_weights/router_xgboost
[mem] peak RSS 1.50 GB — after XGBoost training


## 3. Train per-class FT-Transformer routers

A Feature Tokenizer + Transformer (Gorishniy et al., 2021): each of the 17
numeric features (all but the map-derived `dist_to_drivable_area`, which is
XGBoost-only) is projected to a 64-dim embedding, a learnable `[CLS]` token
is prepended, and 3 pre-norm Transformer encoder layers attend across the
resulting sequence. Trained with BCE loss weighted by the inverse class
imbalance ratio; outputs calibrated via isotonic regression on a held-out
10% split of the training data. See `src/moe/nn_router.py` for the full
architecture.


In [ ]:
from src.moe.features import NN_FEATURE_NAMES
# excludes dist_to_drivable_area, by name not position
nn_feature_names = NN_FEATURE_NAMES  

nn_routers = train_nn_per_class_routers(
    train_df,
    val_df=eval_df,
    model_type="ft_transformer",
    feature_names=nn_feature_names,
    batch_size=NN_BATCH_SIZE,
    use_gpu=USE_GPU_NN,
    inference_device="cpu",
)

NN_OUT = REPO / "model_weights" / "router_nn"
save_nn_per_class_routers(nn_routers, NN_OUT, 
        meta={"feature_names": nn_feature_names})
free_memory("after FT-Transformer training")


2026-08-02 09:59:26 | INFO     | src.moe.nn_router | Training FT_TRANSFORMER router per class on cuda | features=16 | epochs=30 | batch=4096
2026-08-02 09:59:26 | INFO     | src.moe.nn_router | Training car                   : 363530 rows, 33.0% positive


/home/santoshaibox/msaai/capstone/code/direct-data-code/moe-lidar-detection/src/moe/nn_router.py:210: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


2026-08-02 10:00:05 | INFO     | src.moe.nn_router |   epoch  5/30  train_loss=0.2578  val_loss=0.2508  lr=1.00e-03
2026-08-02 10:00:43 | INFO     | src.moe.nn_router |   epoch 10/30  train_loss=0.2510  val_loss=0.2457  lr=1.00e-03
2026-08-02 10:01:21 | INFO     | src.moe.nn_router |   epoch 15/30  train_loss=0.2478  val_loss=0.2455  lr=1.00e-03
2026-08-02 10:02:00 | INFO     | src.moe.nn_router |   epoch 20/30  train_loss=0.2453  val_loss=0.2401  lr=1.00e-03
2026-08-02 10:02:39 | INFO     | src.moe.nn_router |   epoch 25/30  train_loss=0.2426  val_loss=0.2375  lr=1.00e-03
2026-08-02 10:03:18 | INFO     | src.moe.nn_router |   epoch 30/30  train_loss=0.2402  val_loss=0.2360  lr=1.00e-03
2026-08-02 10:03:18 | INFO     | src.moe.nn_router |   Isotonic calibration fitted on 36197 samples (33.2% positive)
2026-08-02 10:03:20 | INFO     | src.moe.nn_router |   └─ val AUC=0.9757  AP=0.9583
2026-08-02 10:03:20 | INFO     | src.moe.nn_router | Training truck                 : 141284 rows, 12.7

/home/santoshaibox/msaai/capstone/code/direct-data-code/moe-lidar-detection/src/moe/nn_router.py:210: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


2026-08-02 10:03:35 | INFO     | src.moe.nn_router |   epoch  5/30  train_loss=0.3956  val_loss=0.4178  lr=1.00e-03
2026-08-02 10:03:50 | INFO     | src.moe.nn_router |   epoch 10/30  train_loss=0.3671  val_loss=0.3845  lr=1.00e-03
2026-08-02 10:04:05 | INFO     | src.moe.nn_router |   epoch 15/30  train_loss=0.3488  val_loss=0.3701  lr=1.00e-03
2026-08-02 10:04:20 | INFO     | src.moe.nn_router |   epoch 20/30  train_loss=0.3304  val_loss=0.3691  lr=5.00e-04
2026-08-02 10:04:35 | INFO     | src.moe.nn_router |   epoch 25/30  train_loss=0.3218  val_loss=0.3599  lr=5.00e-04
2026-08-02 10:04:51 | INFO     | src.moe.nn_router |   epoch 30/30  train_loss=0.3128  val_loss=0.3449  lr=5.00e-04
2026-08-02 10:04:51 | INFO     | src.moe.nn_router |   Isotonic calibration fitted on 13988 samples (12.9% positive)
2026-08-02 10:04:51 | INFO     | src.moe.nn_router |   └─ val AUC=0.9520  AP=0.7688
2026-08-02 10:04:51 | INFO     | src.moe.nn_router | Training trailer               : 47806 rows, 5.3% 

...

2026-08-02 10:08:24 | INFO     | src.moe.nn_router |   epoch  5/30  train_loss=0.3130  val_loss=0.3087  lr=1.00e-03
2026-08-02 10:09:08 | INFO     | src.moe.nn_router |   epoch 10/30  train_loss=0.3055  val_loss=0.3082  lr=1.00e-03
2026-08-02 10:09:53 | INFO     | src.moe.nn_router |   epoch 15/30  train_loss=0.2978  val_loss=0.2972  lr=5.00e-04
2026-08-02 10:10:39 | INFO     | src.moe.nn_router |   epoch 20/30  train_loss=0.2965  val_loss=0.2937  lr=5.00e-04
2026-08-02 10:11:23 | INFO     | src.moe.nn_router |   epoch 25/30  train_loss=0.2910  val_loss=0.2907  lr=2.50e-04
2026-08-02 10:12:08 | INFO     | src.moe.nn_router |   epoch 30/30  train_loss=0.2899  val_loss=0.2911  lr=1.25e-04
2026-08-02 10:12:09 | INFO     | src.moe.nn_router |   Isotonic calibration fitted on 41161 samples (17.7% positive)
2026-08-02 10:12:10 | INFO     | src.moe.nn_router |   └─ val AUC=0.9641  AP=0.7872
2026-08-02 10:12:10 | INFO     | src.moe.nn_router | Training traffic_cone          : 193577 rows, 6.7%

/home/santoshaibox/msaai/capstone/code/direct-data-code/moe-lidar-detection/src/moe/nn_router.py:210: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


2026-08-02 10:12:32 | INFO     | src.moe.nn_router |   epoch  5/30  train_loss=0.4726  val_loss=0.4365  lr=1.00e-03
2026-08-02 10:12:53 | INFO     | src.moe.nn_router |   epoch 10/30  train_loss=0.4647  val_loss=0.4202  lr=1.00e-03
2026-08-02 10:13:14 | INFO     | src.moe.nn_router |   epoch 15/30  train_loss=0.4535  val_loss=0.4317  lr=1.00e-03
2026-08-02 10:13:35 | INFO     | src.moe.nn_router |   epoch 20/30  train_loss=0.4453  val_loss=0.4086  lr=1.00e-03
2026-08-02 10:13:56 | INFO     | src.moe.nn_router |   epoch 25/30  train_loss=0.4380  val_loss=0.4081  lr=1.00e-03
2026-08-02 10:14:13 | INFO     | src.moe.nn_router |   Early stopping at epoch 29 (best val_loss=0.4063 at epoch 24)
2026-08-02 10:14:13 | INFO     | src.moe.nn_router |   Isotonic calibration fitted on 19227 samples (6.7% positive)
2026-08-02 10:14:14 | INFO     | src.moe.nn_router |   └─ val AUC=0.9508  AP=0.4907
2026-08-02 10:14:14 | INFO     | src.moe.nn_router | Training barrier               : 193380 rows, 12.2

/home/santoshaibox/msaai/capstone/code/direct-data-code/moe-lidar-detection/src/moe/nn_router.py:210: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


2026-08-02 10:14:35 | INFO     | src.moe.nn_router |   epoch  5/30  train_loss=0.5225  val_loss=0.5246  lr=1.00e-03
2026-08-02 10:14:56 | INFO     | src.moe.nn_router |   epoch 10/30  train_loss=0.5036  val_loss=0.5210  lr=1.00e-03
2026-08-02 10:15:18 | INFO     | src.moe.nn_router |   epoch 15/30  train_loss=0.4904  val_loss=0.5057  lr=1.00e-03
2026-08-02 10:15:39 | INFO     | src.moe.nn_router |   epoch 20/30  train_loss=0.4811  val_loss=0.4903  lr=5.00e-04
2026-08-02 10:16:00 | INFO     | src.moe.nn_router |   epoch 25/30  train_loss=0.4735  val_loss=0.4879  lr=5.00e-04
2026-08-02 10:16:21 | INFO     | src.moe.nn_router |   epoch 30/30  train_loss=0.4664  val_loss=0.4926  lr=2.50e-04
2026-08-02 10:16:21 | INFO     | src.moe.nn_router |   Early stopping at epoch 30 (best val_loss=0.4879 at epoch 25)
2026-08-02 10:16:21 | INFO     | src.moe.nn_router |   Isotonic calibration fitted on 19210 samples (12.2% positive)
2026-08-02 10:16:22 | INFO     | src.moe.nn_router |   └─ val AUC=0.94

## 4. Router-level evaluation

AUC and Average Precision per class on `eval.csv`, for the XGBoost router,
the FT-Transformer router, and their per-class convex blend
(`combined = lambda * p_xgboost + (1 - lambda) * p_ft_transformer`, with
`lambda` from `configs/moe_final.yaml` → `ensemble_lambda`). This is a
*candidate-level* metric — precision/recall on individual scored boxes,
before NMS or temporal refinement — not the official nuScenes mAP, which
Step 5 computes.


In [6]:
import yaml
from sklearn.metrics import roc_auc_score, average_precision_score
from src.moe.features import NN_FEATURE_NAMES
_NN_FEATURE_IDX = [FEATURE_NAMES.index(f) for f in NN_FEATURE_NAMES]

cfg = yaml.safe_load((REPO / "configs" / "moe_final.yaml").read_text())
lambdas = cfg["ensemble_lambda"]

_CLASS_TO_ID = {c: i for i, c in enumerate(
    ["car", "truck", "trailer", "bus", "construction_vehicle",
     "bicycle", "motorcycle", "pedestrian", "traffic_cone", "barrier"]
)}

rows = []
for cls in NUSCENES_CLASSES:
    sub = eval_df[eval_df["class_id"] == _CLASS_TO_ID[cls]]
    if len(sub) == 0 or sub["label"].sum() == 0:
        continue
    X = sub[FEATURE_NAMES].values.astype(np.float32)
    y = sub["label"].values.astype(int)

    p_xgb = xgb_routers[cls].predict_proba(X)[:, 1]
    p_nn = nn_routers[cls].predict_proba(X[:, _NN_FEATURE_IDX])[:, 1]
    lam = lambdas[cls]
    p_ens = lam * p_xgb + (1 - lam) * p_nn

    rows.append({
        "class": cls,
        "n_eval": len(sub),
        "pos_pct": 100 * y.mean(),
        "xgb_AP": average_precision_score(y, p_xgb),
        "nn_AP": average_precision_score(y, p_nn),
        "ensemble_AP": average_precision_score(y, p_ens),
        "lambda": lam,
    })

results_df = pd.DataFrame(rows).set_index("class")
display(results_df.round(4))

free_memory("Step 4 complete — router-level evaluation")


,n_eval,pos_pct,xgb_AP,nn_AP,ensemble_AP,lambda
class,,,,,,
car,90082,36.6766,0.9637,0.9583,0.9638,0.900
truck,34858,11.5583,0.7825,0.7688,0.7994,0.800
construction_vehicle,15838,0.3536,0.0788,0.0515,0.0752,0.800
bus,6731,23.8003,0.9755,0.9544,0.9744,0.900
trailer,12309,10.5126,0.7830,0.7690,0.8045,0.900
barrier,29815,3.9309,0.5347,0.4158,0.5301,0.974
motorcycle,19402,7.0663,0.8206,0.8036,0.8196,0.800
bicycle,39828,2.4204,0.6848,0.6415,0.6854,0.800
pedestrian,86217,13.0624,0.7698,0.7872,0.7906,0.300


In [ ]:
# Run-mode flags moved to the "Run-mode overrides" 
# cell under Section 1, so a single cell controls them. 
# Setting them here too would silently win, since
# this cell runs later.

## 5. Overfitting / underfitting diagnostics

To check whether either router is over- or under-fitting, this section compares
each model's Average Precision (AP) and AUC on the **training** partition
(which it was fit on) against the **held-out calibration** partition
(`eval.csv`, used only for evaluation everywhere else in this notebook).

- A **large train → eval gap** (train much higher than eval) indicates
  overfitting: the model has partly memorized training rows rather than
  learning a decision boundary that generalizes.
- **Both scores uniformly low, with a small gap** indicates underfitting: the
  model doesn't have enough signal or capacity to separate that class's true
  and false positives.

These are heuristic diagnostics, not hard rules — classes with very few
positive examples (e.g. trailer, construction_vehicle) are noisier at the
class level regardless of fit quality. The isotonic/sigmoid calibration
steps inside each router are fit on held-out slices of the *training*
partition, not on `eval.csv`, so they don't inflate these eval-side numbers.


In [ ]:
from src.moe.features import NN_FEATURE_NAMES
_NN_FEATURE_IDX = [FEATURE_NAMES.index(f) for f in NN_FEATURE_NAMES]

def _class_metrics(df_subset, xgb_router, nn_router, lam):
    X = df_subset[FEATURE_NAMES].values.astype(np.float32)
    y = df_subset["label"].values.astype(int)
    p_xgb = xgb_router.predict_proba(X)[:, 1]
    p_nn = nn_router.predict_proba(X[:, _NN_FEATURE_IDX])[:, 1]
    p_ens = lam * p_xgb + (1 - lam) * p_nn
    return {
        "xgb_AUC": roc_auc_score(y, p_xgb), 
        "xgb_AP": average_precision_score(y, p_xgb),
        "nn_AUC": roc_auc_score(y, p_nn), 
        "nn_AP": average_precision_score(y, p_nn),
        "ens_AUC": roc_auc_score(y, p_ens), 
        "ens_AP": average_precision_score(y, p_ens),
    }


rows = []
for cls in NUSCENES_CLASSES:
    sub_train = train_df[train_df["class_id"] == _CLASS_TO_ID[cls]]
    sub_eval = eval_df[eval_df["class_id"] == _CLASS_TO_ID[cls]]
    if len(sub_eval) == 0 or sub_eval["label"].sum() == 0:
        continue

    lam = lambdas[cls]
    train_m = _class_metrics(
        sub_train, xgb_routers[cls], nn_routers[cls], lam)
    eval_m = _class_metrics(
        sub_eval, xgb_routers[cls], nn_routers[cls], lam)

    row = {"class": cls, "n_train": len(sub_train), "n_eval": len(sub_eval)}
    for model in ["xgb", "nn", "ens"]:
        row[f"{model}_train_AP"] = train_m[f"{model}_AP"]
        row[f"{model}_eval_AP"] = eval_m[f"{model}_AP"]
        row[f"{model}_gap_AP"] = train_m[f"{model}_AP"] - eval_m[f"{model}_AP"]
    rows.append(row)

diag_df = pd.DataFrame(rows).set_index("class")

# Heuristic thresholds -- see markdown above; not hard pass/fail rules.
OVERFIT_GAP_THRESHOLD = 0.15   # train AP this much higher than eval AP
UNDERFIT_AP_THRESHOLD = 0.30   # both train and eval AP below this


def _diagnose(row, model):
    gap = row[f"{model}_gap_AP"]
    train_ap = row[f"{model}_train_AP"]
    eval_ap = row[f"{model}_eval_AP"]
    if gap > OVERFIT_GAP_THRESHOLD:
        return "overfitting risk"
    if (
        train_ap < UNDERFIT_AP_THRESHOLD
        and eval_ap < UNDERFIT_AP_THRESHOLD
    ):
        return "underfitting risk"
    return "OK"


for model in ["xgb", "nn", "ens"]:
    diag_df[f"{model}_diagnosis"] = diag_df.apply(
        lambda r: _diagnose(r, model), axis=1)

cols_order = ["n_train", "n_eval"] + [
    f"{m}_{k}" for m in ["xgb", "nn", "ens"] for k in [
        "train_AP", "eval_AP", "gap_AP", "diagnosis"]
]
display(diag_df[cols_order].round(4))

# This step scores every router over the full ~2M training rows, 
# so it is the heaviest CPU stretch outside Section 7 -- and the 
# last crash landed somewhere between Step 3's save and Section 7, 
# with no marker in between to narrow it.
free_memory("Step 5 complete — train-vs-eval AP diagnostics")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, model, title in zip(
    axes,
    ["xgb", "nn", "ens"],
    ["XGBoost", "FT-Transformer", "Ensemble"],
):
    x = np.arange(len(diag_df))
    width = 0.35
    ax.bar(x - width / 2, diag_df[f"{model}_train_AP"], 
           width, label="train", color="steelblue")
    ax.bar(x + width / 2, diag_df[f"{model}_eval_AP"], 
           width, label="eval", color="seagreen")
    ax.set_xticks(x)
    ax.set_xticklabels(diag_df.index, rotation=45, 
                       ha="right", fontsize=8)
    ax.set_title(title)
    ax.set_ylabel("Average Precision")
    ax.legend(fontsize=8)
fig.suptitle(
    "Train vs. Held-Out Eval AP per Class (gap = overfitting signal)")
plt.tight_layout()
plt.show()

diag_cols = [f"{m}_diagnosis" for m in ["xgb", "nn", "ens"]]
n_overfit = (diag_df[diag_cols] == "overfitting risk").sum().sum()
n_underfit = (diag_df[diag_cols] == "underfitting risk").sum().sum()
n_total = len(diag_df) * len(diag_cols)
print(f"Flags across all classes x models: {n_overfit} overfitting risk, "
      f"{n_underfit} underfitting risk, {n_total - n_overfit - n_underfit} OK "
      f"(out of {n_total} class-model pairs)")

for model, name in [
    ("xgb", "XGBoost"),
    ("nn", "FT-Transformer"),
    ("ens", "Ensemble"),
]:
    flagged = diag_df[
        diag_df[f"{model}_diagnosis"] != "OK"
    ]

    if len(flagged):
        print(f"\n{name} flags:")
        print(
            flagged[
                [
                    f"{model}_train_AP",
                    f"{model}_eval_AP",
                    f"{model}_gap_AP",
                    f"{model}_diagnosis",
                ]
            ].round(4)
        )
free_memory("Step 5b complete — entering Section 6/7")


**What changed since the first version of this diagnostic, and why:**

Fixing this required two independent changes, plus catching a real bug along the way:

1. **XGBoost**: added row/feature subsampling (`subsample=0.8`, `colsample_bytree=0.8`), L2
   regularization (`reg_lambda=2.0`), and per-class early stopping. The early-stopping
   validation set matters a lot here: an internal split carved out of the *training*
   partition was tried first and made things **worse**, not better — every class's
   round count grew to nearly the maximum allowed, because a held-out slice of the
   same 85 training scenes is still more similar to the rest of training than the
   genuinely different 20 calibration scenes are. The real gap is a scene-level
   distribution shift between partitions, not sample-level overfitting, so only a
   validation set drawn from the actually-different distribution can detect it.
   Switching to early stopping directly against `eval.csv` — legitimate here, since
   that partition is this project's designated split "for calibrating post-hoc
   hyperparameters" — fixed this.
2. **FT-Transformer**: added early stopping with best-checkpoint restore, using the
   same held-out 10% slice already used for isotonic calibration.
3. **A real, pre-existing bug, caught by this diagnostic**: `src/moe/nn_router.py`'s
   class list was ordered differently from the `class_id` encoding used everywhere
   else in the pipeline, so 6 of 10 per-class FT-Transformer routers were silently
   trained on the *wrong* class's data (e.g. the router named `"bicycle"` was
   actually fit on pedestrian's rows). Car, truck, bus, and motorcycle were
   unaffected only because their positions happened to coincide between the two
   orderings. This bug is present in the original moeproject repository too, not
   something introduced during porting — worth re-checking there.

**Result:** flagged class/model pairs dropped from 13/30 to 11/30, but the more
meaningful change is severity — most previously-flagged classes now sit at gaps of
0.10–0.20 instead of 0.20–0.34 for XGBoost, and the FT-Transformer's numbers are
now internally consistent (e.g. trailer's gap is 0.009, essentially perfect).

**What's still flagged, and why that's the right call:**

- **`construction_vehicle`** (both models, gap ≈ 0.64–0.67): confirmed as a genuine
  data limitation, not something a training recipe can fix. Its calibration split
  has only 56 positive rows for this class, and BEVFusion — by far its strongest
  expert (59.6% train positive rate) — contributes only 32 rows there vs. 840 in
  training. Two independent model families collapsing on eval AP in exactly the
  same way is strong evidence this is the data, not the model.
- **`barrier`** (gap 0.17–0.25) and **`bicycle`**/**`pedestrian`** (gap ≈ 0.15–0.19):
  meaningfully improved but not fully closed. Further gains here would likely need
  per-class regularization strength (rather than one shared `reg_lambda`/`subsample`
  across all ten classes) or more training data for these specific classes — a
  reasonable next step, not attempted here to avoid overfitting the fix itself to
  this one calibration split.
- **No classes show underfitting** — every class still reaches at least a moderate
  training AP.

**Caveat (unchanged):** these are still candidate-level AP numbers on the
calibration split, before NMS and temporal refinement. A true independent
generalization check would need the project's separate 45-scene held-out test set,
which isn't included in this repo.


## 6. Cross-validation diagnostic: is `construction_vehicle`'s gap real overfitting?

Step 5 flagged a large train→eval gap for `construction_vehicle` (and, to a
lesser extent, `barrier`) on the single calibration split. But that split
turned out to badly under-represent both classes by chance: the calibration
scenes should hold ~23.6% of each class's instances if proportional to their
share of scenes (804/3411 tokens), but `construction_vehicle` landed at only
1.9% and `barrier` at 5.0% — versus ~25–33% for most other classes. With
only 56 `construction_vehicle` positives in eval, a single split can't tell
you whether the gap is *real* overfitting or just an unlucky one-time draw.

This cell answers that with 5-fold **scene-grouped** cross-validation over
the 105 non-test scenes (the 85 router-train + 20 calibration scenes —
the sacred 45-scene test set is never touched here). Each fold trains a
fresh model on 4/5 of the scenes and evaluates on the held-out 1/5, using
the exact same training recipe (`train_xgboost_per_class_routers`) as
production. Scene grouping is preserved via a precomputed
`sample_token -> scene_token` lookup (`training_data/token_to_scene_map.json`)
so this diagnostic never needs the full nuScenes dataset — only the CSVs
already downloaded from Drive.

- If the train→eval gap is **consistently large across all 5 folds**,
  regardless of which scenes are held out, that's real, reproducible
  overfitting.
- If the gap **swings wildly between folds**, that confirms it's dominated
  by which specific handful of scenes/objects happened to land in the eval
  slice — small-sample variance, not a fixable modeling problem.

This is diagnostic only: it doesn't touch `configs/moe_final.yaml` or the
production routers trained in Steps 2–3.


In [ ]:
if not RUN_CV_DIAGNOSTIC:
    print(
        "RUN_CV_DIAGNOSTIC is False — skipping 5-fold CV "
        "(saves ~5× XGBoost retrains)."
    )
else:
    from sklearn.model_selection import GroupKFold
    import logging
    import torch
    from src.moe.xgboost_router import _CLASS_TO_ID

    token_to_scene = json.loads(
        (DATA_DIR.parent / "token_to_scene_map.json").read_text())

    combined_df = pd.concat([train_df, eval_df], ignore_index=True)
    combined_df["scene_token"] = combined_df[
        "sample_token"].map(token_to_scene)
    assert combined_df["scene_token"].notna().all(), (
        "every train/calibration token should have a "
        "scene mapping"
    )
    CV_CLASSES = ["construction_vehicle", "barrier"]
    cv_class_ids = [_CLASS_TO_ID[c] for c in CV_CLASSES]
    cv_subset = combined_df[
        combined_df["class_id"].isin(cv_class_ids)].reset_index(drop=True)

    gkf = GroupKFold(n_splits=5)
    router_logger = logging.getLogger("src.moe.xgboost_router")
    device = (
        "cuda"
        if USE_GPU_XGB and torch.cuda.is_available()
        else "cpu"
    )

    print(
        f"CV XGBoost training device: {device}"
    )

    cv_rows = []
    for fold_i, (train_idx, eval_idx) in enumerate(
        gkf.split(
            cv_subset,
            groups=cv_subset["scene_token"],
        )
    ):
        fold_train = cv_subset.iloc[train_idx]
        fold_eval = cv_subset.iloc[eval_idx]

        router_logger.setLevel(logging.ERROR)
        fold_routers = train_xgboost_per_class_routers(
            fold_train, val_df=fold_eval, feature_names=FEATURE_NAMES,
            calibrated_classes=set(), use_gpu=USE_GPU_XGB,
        )
        router_logger.setLevel(logging.INFO)

        for cls in CV_CLASSES:
            clf = fold_routers.get(cls)
            if clf is None:
                continue
            sub_tr = fold_train[fold_train["class_id"] == _CLASS_TO_ID[cls]]
            sub_ev = fold_eval[fold_eval["class_id"] == _CLASS_TO_ID[cls]]
            y_tr, y_ev = sub_tr["label"].values, sub_ev["label"].values
            if y_ev.sum() == 0:
                continue
            X_tr = sub_tr[FEATURE_NAMES].values.astype(np.float32)
            X_ev = sub_ev[FEATURE_NAMES].values.astype(np.float32)
            ap_tr = average_precision_score(y_tr, 
                clf.predict_proba(X_tr)[:, 1])
            ap_ev = average_precision_score(y_ev, 
                    clf.predict_proba(X_ev)[:, 1])
            cv_rows.append({
                "class": cls, "fold": fold_i,
                "n_train_pos": int(y_tr.sum()), "n_eval_pos": int(y_ev.sum()),
                "train_AP": ap_tr, "eval_AP": ap_ev, "gap": ap_tr - ap_ev,
            })

        del fold_routers
        free_memory()

    cv_df = pd.DataFrame(cv_rows)
    display(cv_df.round(4))


In [ ]:
if not RUN_CV_DIAGNOSTIC:
    print("RUN_CV_DIAGNOSTIC is False — no CV summary.")
else:
    summary = cv_df.groupby("class")["gap"].agg(
        ["mean", "std", "min", "max"])
    print("Gap (train_AP - eval_AP) across 5 scene-grouped folds:\n")
    print(summary.round(4))
    print()
    for cls in CV_CLASSES:
        sub = cv_df[cv_df["class"] == cls]
        spread = sub["gap"].max() - sub["gap"].min()

        verdict = (
            "swings widely -> mostly small-sample variance"
            if spread > 0.3
            else "fairly stable -> likely real overfitting"
        )

        print(
            f"{cls}: gap ranges {sub['gap'].min():.3f} "
            f"to {sub['gap'].max():.3f} across folds "
            f"({verdict})"
        )


In [ ]:
# ── Restart the kernel from code ──────────────────

# Section 7 is the memory-heavy step, so it is best
# entered with a clean kernel.
#
# A restart is the only thing that reliably releases
# the CUDA context and training-time allocations.
# %reset does not.
#
# This is guarded by a flag so "Run All" walks past
# it. Flip the flag to True and run THIS cell alone.
#
# Execution stops here. Nothing below runs until you
# continue manually.
#
# Afterwards, re-run the Setup cell and the Run-mode
# overrides cell, then jump to Section 7. Routers and
# config reload from disk.
#
# On ipykernel 7 (installed here: 7.3.0),
# do_shutdown only flags the kernel to exit.
# The editor notices the dead kernel and brings up a
# new one.
#
# If yours reports "kernel died" instead of
# restarting, restart it from the UI.
RESTART_KERNEL = False

if RESTART_KERNEL:
    import IPython

    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print(
        "RESTART_KERNEL is False — set it to True and "
        "run this cell to restart."
    )

## 7. *(Optional, advanced)* Full pipeline + official nuScenes evaluation

This section reproduces the project's headline result end to end: ensemble
blend → soft-temperature gate → per-class thresholds → class-aware NMS →
temporal refinement (orphan penalty + interpolation) → official nuScenes
mAP/NDS. It requires two things this notebook doesn't need above:

1. **Expert prediction JSONs** for the five frozen detectors — download via
   [`predictions/README.md`](../predictions/README.md) (~1.6 GB) and place
   under `predictions/<expert_name>/predictions.json`.
2. **nuScenes metadata** — *not* the full 306 GB sensor dataset. Only the
   `v1.0-trainval` JSON tables and `maps/` directory are needed (no camera
   or LiDAR sensor blobs, no map-expansion vector pack): this pipeline's
   features come from box geometry and inter-expert agreement, not raw
   point clouds. Register at nuscenes.org, download the "Metadata" and
   "Map expansion" (base, not the full pack) assets for the trainval split,
   and extract them to `data/nuscenes/`.

Set `RUN_FULL_PIPELINE = True` below once both are in place. Left `False`
by default so the rest of the notebook runs standalone.


In [ ]:
if RUN_FULL_PIPELINE:
    from nuscenes import NuScenes

    from src.io.load_predictions import load_nuscenes_predictions
    from src.io.save_predictions import save_nuscenes_predictions
    from src.moe.pipeline import build_ensemble_routers, run_final_pipeline
    from src.moe.router_dataset import build_token_to_mask, warm_map_masks
    from src.moe.features import _DRIVABLE_AREA_DILATION_LEVELS_M
    from src.evaluation.evaluate_nuscenes import evaluate_submission
    import os

    free_memory("entering Section 7")

    # Also reload the config if Step 4 didn't run in this kernel. With
    # EVAL_SPLIT="test" this cell then needs nothing from Steps 1-6.
    if "cfg" not in globals():
        import yaml
        cfg = yaml.safe_load((
            REPO / "configs" / "moe_final.yaml").read_text())
        lambdas = cfg["ensemble_lambda"]

    # Point this at your nuScenes metadata root (must contain v1.0-trainval/).
    NUSCENES_ROOT = Path(os.environ.get("NUSCENES_ROOT", 
                REPO / "data" / "nuscenes"))
    if not NUSCENES_ROOT.exists():
        raise FileNotFoundError(
            f"nuScenes metadata not found at {NUSCENES_ROOT}. Place it at "
            "data/nuscenes/ or set the NUSCENES_ROOT environment variable."
        )

    EXPERTS = ["centerpoint", "centerpoint_pillar",
                "pointpillars", "ssn", "bevfusion_lidar"]

    # Which frames to score. The reported mAP=0.6179 / NDS=0.6761 is on the
    # held-out test scenes; the calibration split is only a fast sanity check
    # and scores much lower (see the EVAL_SPLIT note in Setup).
    if EVAL_SPLIT == "test":
        _split = json.loads((
            REPO / "training_data" / "token_split_3way.json").read_text())
        sample_tokens = _split["test_tokens"]
    else:
        sample_tokens = eval_df["sample_token"].unique().tolist()
    sample_token_set = set(sample_tokens)
    print(f"Scoring EVAL_SPLIT='{EVAL_SPLIT}' — {len(sample_tokens):,} keyframes")

    # Load one expert at a time, keeping only the tokens this run needs.
    # Measured: 1.9 GB for the 804 calibration frames vs 3.9 GB for all 6,019.
    predictions = {}
    for name in EXPERTS:
        predictions[name] = load_nuscenes_predictions(
            path=REPO / "predictions" / name / "predictions.json",
            model_name=name,
            frame="global",
            sample_tokens=sample_token_set,
        )
        free_memory(f"loaded expert {name}")

    # Reload from disk when this is a fresh kernel (the recommended way to run
    # Section 7: Steps 2-3 already wrote these, and not holding training state
    # resident is the whole point of splitting the sessions).
    if "xgb_routers" not in globals() or "nn_routers" not in globals():
        from src.moe.nn_router import load_nn_per_class_routers
        xgb_routers = load_xgboost_per_class_routers(
            REPO / "model_weights" / "router_xgboost")
        nn_routers = load_nn_per_class_routers(
            REPO / "model_weights" / "router_nn")

    ensemble_routers = build_ensemble_routers(xgb_routers, nn_routers, lambdas)

    nusc = NuScenes(version="v1.0-trainval", 
                    dataroot=str(NUSCENES_ROOT), verbose=False)
    free_memory("nuScenes metadata loaded (~8 GB)")

    mask_by_token = build_token_to_mask(nusc, sample_tokens)
    # Computing the dilated masks up front is not optional memory-wise: feature
    # extraction needs them either way. Doing it here gets the gc.collect()
    # between maps, which bounds the transient distanceTransform peak instead of
    # letting it stack on top of the live prediction data during inference.
    warm_map_masks(mask_by_token, _DRIVABLE_AREA_DILATION_LEVELS_M)
    free_memory("map masks dilated (biggest single spike, measured ~24 GB peak)")

    final_predictions = run_final_pipeline(
        predictions, ensemble_routers, sample_tokens, 
        cfg, nusc, mask_by_token=mask_by_token,
    )
    # `predictions` stays alive on purpose — Section 8b renders each expert.
    free_memory("pipeline complete")

    submission_path = REPO / "outputs" / "submission.json"
    submission_path.parent.mkdir(exist_ok=True)
    save_nuscenes_predictions(final_predictions, submission_path)

    result = evaluate_submission(
        submission_path,
        nuscenes_root=NUSCENES_ROOT,
        version="v1.0-trainval",
        split="val",
        sample_tokens=sample_tokens,
    )
    print(f"mAP={result['map']:.4f}  NDS={result['nds']:.4f}")
    pd.Series(result["summary"]).round(4)
else:
    print("RUN_FULL_PIPELINE is False — skipping. See the markdown cell above for requirements.")


## 8. *(Optional, advanced)* Visualize predictions vs. ground truth

Renders a few real test-set frames in bird's-eye view: the LiDAR point
cloud, this pipeline's actual predicted boxes (solid, colour-coded by
class, with confidence scores), and ground truth (white dashed outlines)
for direct visual comparison. Uses `src/utils/bev_viz.py`.

Requires `RUN_FULL_PIPELINE = True` above (so `final_predictions` exists)
**and** the raw nuScenes LiDAR sweeps (`data/nuscenes/samples/LIDAR_TOP/`)
— not part of the Drive-hosted CSVs/predictions/weights, since the full
dataset is ~306GB. See `docs/expert_regeneration.md`.

Example output is checked into `docs/figures/inference_examples/` from a
run against the true held-out test set (mAP=0.6179 configuration).


In [ ]:
if RUN_FULL_PIPELINE:
    import matplotlib
    from nuscenes.eval.common.loaders import load_gt_of_sample_tokens
    from nuscenes.eval.detection.data_classes import DetectionBox as NuScDetectionBox
    from src.utils.bev_viz import read_pcd_bin, global_box_to_ego, render_bev_frame

    MIN_DISPLAY_SCORE = 0.35  # display-only filter for readability; not the eval threshold
    N_FRAMES_TO_SHOW = 3

    # Pick the busiest frames among what Step 7 just ran as interesting examples.
    candidate_tokens = sorted(
        final_predictions.keys(), key=lambda t: len(final_predictions[t]), reverse=True
    )[:N_FRAMES_TO_SHOW]
    gt_boxes_all = load_gt_of_sample_tokens(nusc, candidate_tokens, NuScDetectionBox, verbose=False)

    for token in candidate_tokens:
        sample = nusc.get("sample", token)
        sd = nusc.get("sample_data", sample["data"]["LIDAR_TOP"])
        ego_pose = nusc.get("ego_pose", sd["ego_pose_token"])
        points = read_pcd_bin(NUSCENES_ROOT / sd["filename"])

        pred_boxes = [
            b.to_nuscenes_dict() for b in final_predictions[token]
            if b.detection_score >= MIN_DISPLAY_SCORE
        ]
        pred_boxes_ego = [global_box_to_ego(b, ego_pose) for b in pred_boxes]

        gt_boxes = [
            {"translation": list(b.translation), "size": list(b.size),
             "rotation": list(b.rotation), "detection_name": b.detection_name}
            for b in gt_boxes_all[token]
        ]
        gt_boxes_ego = [global_box_to_ego(b, ego_pose) for b in gt_boxes]

        scene = nusc.get("scene", sample["scene_token"])
        fig = render_bev_frame(
            points, pred_boxes_ego, gt_boxes_ego,
            title=f"{scene['name']} | {len(pred_boxes_ego)} predictions, {len(gt_boxes_ego)} GT boxes",
        )
        plt.show()
else:
    print("RUN_FULL_PIPELINE is False — skipping. See the markdown cell above for requirements.")


### 8b. Each expert alone vs. the learned ensemble

The same idea, but as a grid: each of the 5 frozen experts' raw
predictions (before any router/fusion) next to the final MoE ensemble
output, all against the same point cloud and ground truth. This is the
most direct visual answer to "why ensemble at all" -- individual experts
routinely miss objects the ensemble catches (see the boxes-shown count per
panel), because each was trained with a different architecture/voxelization
and none of them is systematically better across every object type and
range.


In [ ]:
if RUN_FULL_PIPELINE:
    from src.utils.bev_viz import render_expert_comparison_grid

    EXPERT_DISPLAY_NAMES = {
        "centerpoint": "CenterPoint (voxel)",
        "centerpoint_pillar": "CenterPoint (pillar)",
        "pointpillars": "PointPillars",
        "ssn": "SSN",
        "bevfusion_lidar": "BEVFusion-LiDAR",
    }
    EXPERT_MIN_SCORE = 0.3

    for token in candidate_tokens:
        sample = nusc.get("sample", token)
        sd = nusc.get("sample_data", sample["data"]["LIDAR_TOP"])
        ego_pose = nusc.get("ego_pose", sd["ego_pose_token"])
        points = read_pcd_bin(NUSCENES_ROOT / sd["filename"])

        gt_boxes = [
            {"translation": list(b.translation), "size": list(b.size),
             "rotation": list(b.rotation), "detection_name": b.detection_name}
            for b in gt_boxes_all[token]
        ]
        gt_boxes_ego = [global_box_to_ego(b, ego_pose) for b in gt_boxes]

        pred_boxes_by_source = {}
        for name, display_name in EXPERT_DISPLAY_NAMES.items():
            raw = [
                b.to_nuscenes_dict() for b in predictions[name].get(token, [])
                if b.detection_score >= EXPERT_MIN_SCORE
            ]
            pred_boxes_by_source[display_name] = [global_box_to_ego(b, ego_pose) for b in raw]

        ensemble_raw = [
            b.to_nuscenes_dict() for b in final_predictions[token]
            if b.detection_score >= MIN_DISPLAY_SCORE
        ]
        pred_boxes_by_source["Final MoE Ensemble"] = [global_box_to_ego(b, ego_pose) for b in ensemble_raw]

        scene = nusc.get("scene", sample["scene_token"])
        fig = render_expert_comparison_grid(
            points, pred_boxes_by_source, gt_boxes_ego,
            suptitle=f"{scene['name']}  |  {len(gt_boxes_ego)} ground-truth objects  |  each expert alone vs. the learned ensemble",
        )
        plt.show()


## 9. Reference results

**This repo's confirmed result** (retrained routers: early stopping +
regularization, the nn_router.py class-order bug fixed, and the
zero-importance `class_id` feature dropped from the model input), run
through the full pipeline in `configs/moe_final.yaml` and evaluated with
the official nuScenes evaluator on the 45 held-out test scenes (1,804
keyframes, `test_tokens` in `training_data/token_split_3way.json`) —
never touched during router training or threshold/lambda calibration:

| Metric | Value |
|---|---|
| mAP | 0.6179 |
| NDS | 0.6761 |

**For context**, the original project's result before these fixes (same
test scenes, same pipeline config, unregularized routers with the class-id
feature and the class-order bug both still present):

| Metric | Value |
|---|---|
| mAP | 0.6152 |
| NDS | 0.6694 |
| Gap to best single expert (BEVFusion-LiDAR, mAP 0.5844) | +3.08 pp |

To reproduce these numbers, keep `EVAL_SPLIT = "test"` in the run-mode
overrides cell (Section 1) when running Section 7. The other setting,
`"calibration"`, scores `eval.csv`'s 804 tokens instead — a much faster
sanity check, but those are the frames used to tune the thresholds and
lambdas, and they are short on `construction_vehicle` and `barrier`, so the
mAP reads around 0.53 and is not comparable to the table above.

Two things must also match, because both change the trained routers rather
than just the evaluation: `NN_BATCH_SIZE` has to be `4096` (the value in
`configs/moe_final.yaml` — batch size determines which rows share a gradient
step, and training at 1024 measured 0.6157 instead), and the FT-Transformer
must see the 16 `NN_FEATURE_NAMES` features rather than all 17. Expect
agreement to a few parts in ten thousand rather than exactly: GPU reductions
are not bit-reproducible run to run.


## 10. Every expert vs. the ensemble, scored the same way

Section 8b compares experts *visually*, on three frames. This section does it
numerically, over every frame Section 7 scored, with the official evaluator.

The comparison is only meaningful if nothing varies except the fusion, so each
expert is scored on the identical token set, with the identical evaluator
config, as the MoE row. Each expert's raw boxes are re-exported from the
`predictions` dict Section 7 still holds — already filtered to those tokens —
rather than re-read from the full-val JSONs on disk, which cost several GB to
parse and would pull in frames the MoE was never scored on.

Requires `RUN_FULL_PIPELINE = True` and Section 7 to have run in this kernel
(it needs `predictions`, `result`, `nusc`, and `sample_tokens`). Expect this to
take roughly as long as five Section 7 evaluations, since ground truth is
re-loaded per expert.

In [ ]:
if RUN_FULL_PIPELINE:
    import tempfile

    from src.io.save_predictions import save_nuscenes_predictions
    from src.evaluation.evaluate_nuscenes import evaluate_submission

    MOE_LABEL = "MoE ensemble"

    # `nusc=nusc` matters here: without it each call would re-parse the ~8 GB
    # trainval tables, five times over, for tables already in memory.
    expert_results = {}
    with tempfile.TemporaryDirectory() as tmp_dir:
        for name in EXPERTS:
            expert_submission = Path(tmp_dir) / f"{name}.json"
            save_nuscenes_predictions(predictions[name], expert_submission)
            expert_results[name] = evaluate_submission(
                expert_submission, sample_tokens=sample_tokens, nusc=nusc,
            )
            free_memory(f"evaluated expert {name}")

    rows = [
        {"model": name, "mAP": r["map"], "NDS": r["nds"], **r["per_class_ap"]}
        for name, r in expert_results.items()
    ]
    rows.append(
        {"model": MOE_LABEL, "mAP": result["map"], "NDS": result["nds"],
         **result["per_class_ap"]}
    )
    comparison_df = pd.DataFrame(rows).set_index("model").sort_values("mAP")

    print(f"nuScenes evaluation on {len(sample_tokens):,} '{EVAL_SPLIT}' keyframes\n")
    display(comparison_df[["mAP", "NDS"]].round(4))

    best_expert = comparison_df.drop(index=MOE_LABEL)["mAP"].idxmax()
    best_map = comparison_df.loc[best_expert, "mAP"]
    moe_map = comparison_df.loc[MOE_LABEL, "mAP"]
    best_nds = comparison_df.loc[best_expert, "NDS"]
    moe_nds = comparison_df.loc[MOE_LABEL, "NDS"]
    print(
        f"\nBest single expert: {best_expert} — mAP {best_map:.4f}, NDS {best_nds:.4f}\n"
        f"Ensemble:           mAP {moe_map:.4f}, NDS {moe_nds:.4f}\n"
        f"Gain from fusion:   {100 * (moe_map - best_map):+.2f} pp mAP, "
        f"{100 * (moe_nds - best_nds):+.2f} pp NDS"
    )

    # Per class, is the ensemble beating the best expert available for that class,
    # or just the best expert overall? Those differ -- no expert wins everywhere,
    # which is the premise of routing per class in the first place.
    per_class_df = comparison_df.drop(columns=["mAP", "NDS"]).T
    per_class_df["best expert"] = per_class_df[EXPERTS].max(axis=1)
    per_class_df["which expert"] = per_class_df[EXPERTS].idxmax(axis=1)
    per_class_df["MoE - best"] = per_class_df[MOE_LABEL] - per_class_df["best expert"]
    print("\nPer-class AP: ensemble vs. the strongest expert for that class\n")
    display(
        per_class_df[["which expert", "best expert", MOE_LABEL, "MoE - best"]]
        .sort_values("MoE - best")
        .round(4)
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    bar_colors = ["#c44e52" if m == MOE_LABEL else "#8c9bb0" for m in comparison_df.index]
    ax.barh(comparison_df.index, comparison_df["mAP"], color=bar_colors)
    for y, value in enumerate(comparison_df["mAP"]):
        ax.text(value + 0.005, y, f"{value:.4f}", va="center", fontsize=9)
    ax.set_xlim(0, comparison_df["mAP"].max() * 1.15)
    ax.set_xlabel("mAP")
    ax.set_title(f"Frozen experts vs. learned ensemble — {len(sample_tokens):,} {EVAL_SPLIT} keyframes")
    plt.tight_layout()
    plt.show()
else:
    print("RUN_FULL_PIPELINE is False — skipping. Section 7 must run first.")